In [ ]:
# Install yfinance if not installed
# pip install yfinance

# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN

# Download Google stock data online
data = yf.download("GOOG", start="2015-01-01", end="2024-01-01")

# Use Open price
dataset = data[['Open']]

# Convert to array
training_set = dataset.values

# Feature Scaling
sc = MinMaxScaler(feature_range=(0,1))
training_set_scaled = sc.fit_transform(training_set)

# Create training data
X_train = []
y_train = []

for i in range(60, len(training_set_scaled)):
    X_train.append(training_set_scaled[i-60:i, 0])
    y_train.append(training_set_scaled[i, 0])

X_train = np.array(X_train)
y_train = np.array(y_train)

# Reshape for RNN
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))

# Build RNN model
model = Sequential()

model.add(SimpleRNN(units=50, activation='tanh',
                    input_shape=(X_train.shape[1],1)))

model.add(Dense(units=1))

# Compile model
model.compile(optimizer='adam', loss='mean_squared_error')

# Train model
model.fit(X_train, y_train, epochs=20, batch_size=32)

# Predict values
predicted_stock_price = model.predict(X_train)

# Convert back to original values
predicted_stock_price = sc.inverse_transform(predicted_stock_price)

# Plot graph
plt.plot(dataset.values[60:], color='red',
         label='Real Google Stock Price')

plt.plot(predicted_stock_price, color='blue',
         label='Predicted Google Stock Price')

plt.title('Google Stock Price Prediction using RNN')
plt.xlabel('Time')
plt.ylabel('Google Stock Price')
plt.legend()
plt.show()